In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot

In [2]:
root_path = Path("../ForChiara-OldData/alpha1p47/alpha1p47_0um_Cytoplasm_900.root")
root_file = uproot.open(root_path)

primary_tree = root_file["Ntuples/primary_source"]
damage_tree = root_file["Ntuples/damage"]
classification_tree = root_file["Ntuples/classification"]

primary_tree.num_entries, damage_tree.num_entries, classification_tree.num_entries

(900, 52865, 259)

In [3]:
primary_tree.keys(), damage_tree.keys()

(['Primary',
  'Energy',
  'PosX_um',
  'PosY_um',
  'PosZ_um',
  'MomX',
  'MomY',
  'MomZ',
  'StopPosX_um',
  'StopPosY_um',
  'StopPosZ_um',
  'TraLen_cell_um',
  'TraLen_chro_um'],
 ['Event',
  'Primary',
  'Energy',
  'TypeClassification',
  'SourceClassification',
  'Position_x_um',
  'Position_y_um',
  'Position_z_um',
  'Size_nm',
  'FragmentLength',
  'BaseDamage',
  'StrandDamage',
  'DirectBreaks',
  'IndirectBreaks',
  'EaqBaseHits',
  'EaqStrandHits',
  'OHBaseHits',
  'OHStrandHits',
  'HBaseHits',
  'HStrandHits',
  'EnergyDeposited_eV',
  'InducedBreaks',
  'Chain',
  'Strand',
  'BasePair',
  'Name',
  'TypeClassificationInt',
  'SourceClassificationInt'])

In [4]:
requested_primary_columns = [
    "PosX_um",
    "PosY_um",
    "PosZ_um",
    "MomX",
    "MomY",
    "MomZ",
    "StopPosX_um",
    "StopPosY_um",
    "StopPosZ_um",
    "TraLen_cell_um",
    "TraLen_chro_um",
    "EventID",  # newer MolecularBNCT source
    "Event",    # older ROOT files
    "distance",
]
damage_columns = [
    "Event",
    "Position_x_um",
    "Position_y_um",
    "Position_z_um",
    "EnergyDeposited_eV",
]

primary_columns = [column for column in requested_primary_columns if column in primary_tree.keys()]
missing_primary_columns = sorted(set(requested_primary_columns) - set(primary_columns))

# One row per primary particle/track.
primary_df = primary_tree.arrays(primary_columns, library="pd").reset_index(drop=True)
primary_df.insert(0, "Particle_ID", np.arange(len(primary_df)))

# One row per damage step. Event is local/reused in this stacked file, so do not use it alone as a join key.
damage_df = damage_tree.arrays(damage_columns, library="pd").reset_index(drop=True)

missing_primary_columns, primary_df.head(), damage_df.head()

(['Event', 'EventID', 'distance'],
    Particle_ID   PosX_um   PosY_um   PosZ_um      MomX      MomY      MomZ  \
 0            0 -0.180278  2.103225 -2.652770 -0.762457  0.287191  0.579811   
 1            1  3.598930 -1.633236  0.881580 -0.059075 -0.363542 -0.929703   
 2            2 -2.811010  2.438439  2.647914  0.732676 -0.249092  0.633355   
 3            3  2.617579 -0.004172  3.586020  0.083778  0.044656 -0.995483   
 4            4 -3.417772  1.138756  1.924982  0.714452  0.469036  0.519195   
 
    StopPosX_um  StopPosY_um  StopPosZ_um  TraLen_cell_um  TraLen_chro_um  
 0    -6.198528     3.638295     1.995732        7.757864        0.000000  
 1     3.455261    -4.676111    -6.256297        7.919075        0.000000  
 2     2.310123     0.398453     8.100631        7.794049        0.000000  
 3     2.958992     0.342192    -4.370436        7.862313        1.870643  
 4     2.157638     5.555915     5.087893        7.943326        0.000000  ,
    Event  Position_x_um  Positi

In [5]:
def add_damage_chunk_id(damage_df):
    """Split damage rows into consecutive chunks where the local Event value is constant."""
    local_event = damage_df["Event"].astype(int)
    chunk_start = local_event.ne(local_event.shift(fill_value=local_event.iloc[0]))
    damage_chunk_id = chunk_start.cumsum().astype(int)

    damage_with_chunks = damage_df.copy()
    damage_with_chunks["DamageChunk_ID"] = damage_chunk_id
    return damage_with_chunks


damage_with_chunks = add_damage_chunk_id(damage_df)
event_chunks = (
    damage_with_chunks
    .groupby("DamageChunk_ID", as_index=False)
    .agg(
        Event=("Event", "first"),
        FirstDamageRow=("Event", lambda s: s.index.min()),
        LastDamageRow=("Event", lambda s: s.index.max()),
        DamageRows=("Event", "size"),
        EnergyDeposited_eV=("EnergyDeposited_eV", "sum"),
    )
)

event_chunks.head(), event_chunks.shape, len(primary_df)

(   DamageChunk_ID  Event  FirstDamageRow  LastDamageRow  DamageRows  \
 0               0      6               0             55          56   
 1               1     24              56            327         272   
 2               2     15             328            633         306   
 3               3     21             634            865         232   
 4               4     12             866           1011         146   
 
    EnergyDeposited_eV  
 0          350.737986  
 1         2511.439917  
 2         3646.211159  
 3         1157.554972  
 4          956.318341  ,
 (259, 6),
 900)

In [6]:
def assign_damage_chunks_to_primary_tracks(primary_df, damage_with_chunks):
    """Map each recovered damage chunk to the nearest primary-source track.

    Entry order recovers DamageChunk_ID, but not the original Particle_ID when
    zero-damage primaries are absent from the damage tree. The position match is
    therefore needed only to place each non-empty damage chunk back onto the 900
    primary_source rows.
    """
    starts = primary_df[["PosX_um", "PosY_um", "PosZ_um"]].to_numpy(float)
    stops = primary_df[["StopPosX_um", "StopPosY_um", "StopPosZ_um"]].to_numpy(float)
    segments = stops - starts
    segment_lengths_sq = np.sum(segments**2, axis=1)

    assignments = []
    for damage_chunk_id, chunk in damage_with_chunks.groupby("DamageChunk_ID", sort=True):
        points = chunk[["Position_x_um", "Position_y_um", "Position_z_um"]].to_numpy(float)
        best_particle_id = -1
        best_score = np.inf
        best_min_distance = np.inf

        for particle_id, (start, segment, length_sq) in enumerate(
            zip(starts, segments, segment_lengths_sq)
        ):
            point_vectors = points - start
            if length_sq == 0:
                dists = np.linalg.norm(point_vectors, axis=1)
            else:
                t = np.clip(point_vectors @ segment / length_sq, 0.0, 1.0)
                closest = start + t[:, None] * segment
                dists = np.linalg.norm(points - closest, axis=1)

            score = float(np.median(dists))
            min_distance = float(np.min(dists))
            if score < best_score:
                best_score = score
                best_min_distance = min_distance
                best_particle_id = particle_id

        assignments.append(
            {
                "DamageChunk_ID": damage_chunk_id,
                "Particle_ID": best_particle_id,
                "ChunkMedianDistanceToTrack_um": best_score,
                "ChunkMinDistanceToTrack_um": best_min_distance,
            }
        )

    chunk_assignment = pd.DataFrame(assignments)
    return damage_with_chunks.merge(chunk_assignment, on="DamageChunk_ID", how="left")


damage_with_particle = assign_damage_chunks_to_primary_tracks(primary_df, damage_with_chunks)

particle_energy = (
    damage_with_particle.groupby("Particle_ID", as_index=False, sort=True)["EnergyDeposited_eV"]
    .sum()
    .rename(columns={"EnergyDeposited_eV": "TotalEnergyDeposited_eV"})
)

primary_energy_df = primary_df.merge(particle_energy, on="Particle_ID", how="left")
primary_energy_df["TotalEnergyDeposited_eV"] = primary_energy_df["TotalEnergyDeposited_eV"].fillna(0.0)

primary_energy_df.head(), damage_with_particle[["DamageChunk_ID", "Event", "Particle_ID", "ChunkMedianDistanceToTrack_um"]].drop_duplicates().head()

(   Particle_ID   PosX_um   PosY_um   PosZ_um      MomX      MomY      MomZ  \
 0            0 -0.180278  2.103225 -2.652770 -0.762457  0.287191  0.579811   
 1            1  3.598930 -1.633236  0.881580 -0.059075 -0.363542 -0.929703   
 2            2 -2.811010  2.438439  2.647914  0.732676 -0.249092  0.633355   
 3            3  2.617579 -0.004172  3.586020  0.083778  0.044656 -0.995483   
 4            4 -3.417772  1.138756  1.924982  0.714452  0.469036  0.519195   
 
    StopPosX_um  StopPosY_um  StopPosZ_um  TraLen_cell_um  TraLen_chro_um  \
 0    -6.198528     3.638295     1.995732        7.757864        0.000000   
 1     3.455261    -4.676111    -6.256297        7.919075        0.000000   
 2     2.310123     0.398453     8.100631        7.794049        0.000000   
 3     2.958992     0.342192    -4.370436        7.862313        1.870643   
 4     2.157638     5.555915     5.087893        7.943326        0.000000   
 
    TotalEnergyDeposited_eV  
 0                 0.000000  


In [7]:
primary_energy_df.to_csv("alpha1p47_0um_Cytoplasm_900_total_edep_per_primary.csv", index=False)
primary_energy_df[["Particle_ID", "TotalEnergyDeposited_eV"]].describe()

,Particle_ID,TotalEnergyDeposited_eV
count,900.000000,900.000000
mean,449.500000,562.947374
std,259.951919,1145.767624
min,0.000000,0.000000
25%,224.750000,0.000000
50%,449.500000,0.000000
75%,674.250000,0.000000
max,899.000000,7578.561046


In [8]:
getallparticle_path = root_path.with_name("All" + root_path.stem + ".csv")
getallparticle_df = pd.read_csv(getallparticle_path)

classification_columns = ["SSB", "DSB", "DSBp", "DSBpp"]
getallparticle_damaged_ids = set(
    getallparticle_df.loc[
        getallparticle_df[classification_columns].ne(0).any(axis=1),
        "Particle_ID",
    ].astype(int)
)

energy_assigned_ids = set(damage_with_particle["Particle_ID"].astype(int).unique())

comparison = {
    "GetAllParticle nonzero-classification particles": len(getallparticle_damaged_ids),
    "Energy-mapped damage particles": len(energy_assigned_ids),
    "IDs in both": len(getallparticle_damaged_ids & energy_assigned_ids),
    "Only in GetAllParticle": sorted(getallparticle_damaged_ids - energy_assigned_ids),
    "Only in energy mapping": sorted(energy_assigned_ids - getallparticle_damaged_ids),
}

comparison

{'GetAllParticle nonzero-classification particles': 254,
 'Energy-mapped damage particles': 226,
 'IDs in both': 218,
 'Only in GetAllParticle': [28,
  39,
  43,
  46,
  60,
  68,
  121,
  131,
  159,
  172,
  209,
  219,
  246,
  254,
  270,
  272,
  301,
  304,
  305,
  317,
  318,
  326,
  359,
  403,
  451,
  468,
  563,
  564,
  566,
  667,
  702,
  788,
  795,
  798,
  886,
  896],
 'Only in energy mapping': [np.int64(83),
  np.int64(105),
  np.int64(183),
  np.int64(266),
  np.int64(312),
  np.int64(505),
  np.int64(677),
  np.int64(894)]}

In [9]:
def get_guilty_particles_like_getallparticle(primary_df, damage_df, threshold_um=0.05):
    starts = primary_df[["PosX_um", "PosY_um", "PosZ_um"]].to_numpy(float)
    stops = primary_df[["StopPosX_um", "StopPosY_um", "StopPosZ_um"]].to_numpy(float)
    points = damage_df[["Position_x_um", "Position_y_um", "Position_z_um"]].to_numpy(float)

    min_dists = np.full(len(points), np.inf)
    assigned_particle_ids = np.full(len(points), -1, dtype=int)

    for particle_id, (start, stop) in enumerate(zip(starts, stops)):
        segment = stop - start
        point_vectors = points - start
        segment_length_sq = np.sum(segment**2)

        if segment_length_sq == 0:
            dists = np.linalg.norm(point_vectors, axis=1)
        else:
            t = np.clip(point_vectors @ segment / segment_length_sq, 0.0, 1.0)
            closest = start + t[:, None] * segment
            dists = np.linalg.norm(points - closest, axis=1)

        closer = dists < min_dists
        min_dists[closer] = dists[closer]
        assigned_particle_ids[closer] = particle_id

    valid_mask = min_dists <= threshold_um
    guilty_ids = np.unique(assigned_particle_ids[valid_mask])
    guilty_ids = guilty_ids[guilty_ids != -1]
    return guilty_ids, assigned_particle_ids, min_dists


def tune_threshold_like_getallparticle(primary_df, damage_df, target_count, threshold_um=0.05):
    guilty_ids, particle_map, dists = get_guilty_particles_like_getallparticle(
        primary_df, damage_df, threshold_um=threshold_um
    )

    while len(guilty_ids) < target_count:
        threshold_um += 0.01
        guilty_ids, particle_map, dists = get_guilty_particles_like_getallparticle(
            primary_df, damage_df, threshold_um=threshold_um
        )
        while len(guilty_ids) > target_count:
            threshold_um -= 0.001
            guilty_ids, particle_map, dists = get_guilty_particles_like_getallparticle(
                primary_df, damage_df, threshold_um=threshold_um
            )
            if len(guilty_ids) <= target_count:
                break
            while len(guilty_ids) < target_count:
                threshold_um += 0.0001
                guilty_ids, particle_map, dists = get_guilty_particles_like_getallparticle(
                    primary_df, damage_df, threshold_um=threshold_um
                )
                if len(guilty_ids) >= target_count:
                    break

    return threshold_um, guilty_ids, particle_map, dists


classification_df = classification_tree.arrays(classification_columns, library="pd")
threshold_um, guilty_ids, particle_map, dists = tune_threshold_like_getallparticle(
    primary_df,
    damage_df,
    target_count=len(classification_df),
    threshold_um=0.05,
)

getall_style_chunk_energy = event_chunks[["DamageChunk_ID", "Event", "EnergyDeposited_eV"]].copy()
if len(getall_style_chunk_energy) != len(guilty_ids):
    raise ValueError(
        f"Found {len(getall_style_chunk_energy)} damage chunks but {len(guilty_ids)} "
        "GetAllParticle-style guilty IDs; cannot assign chunk energies by order."
    )

getall_style_chunk_energy["Particle_ID"] = guilty_ids
getall_style_chunk_energy = getall_style_chunk_energy.rename(
    columns={"EnergyDeposited_eV": "TotalEnergyDeposited_eV"}
)

primary_energy_getall_style_df = primary_df.merge(
    getall_style_chunk_energy[["Particle_ID", "TotalEnergyDeposited_eV"]],
    on="Particle_ID",
    how="left",
)
primary_energy_getall_style_df["TotalEnergyDeposited_eV"] = primary_energy_getall_style_df[
    "TotalEnergyDeposited_eV"
].fillna(0.0)

getall_style_ids = set(map(int, guilty_ids))
comparison_getall_style = {
    "classification entries": len(classification_df),
    "damage chunks": len(event_chunks),
    "GetAllParticle-style guilty IDs": len(getall_style_ids),
    "threshold_um": threshold_um,
    "IDs in both exported nonzero and GetAllParticle-style": len(
        getallparticle_damaged_ids & getall_style_ids
    ),
    "Only in exported nonzero": sorted(getallparticle_damaged_ids - getall_style_ids),
    "Only in GetAllParticle-style": sorted(getall_style_ids - getallparticle_damaged_ids),
}

comparison_getall_style

{'classification entries': 259,
 'damage chunks': 259,
 'GetAllParticle-style guilty IDs': 259,
 'threshold_um': 0.09599999999999999,
 'IDs in both exported nonzero and GetAllParticle-style': 254,
 'Only in exported nonzero': [],
 'Only in GetAllParticle-style': [183, 266, 369, 505, 894]}

In [10]:
def add_track_origin_geometry(primary_df, damaging_primary_ids, nucleus_radius_um=None):
    """Add closest-approach-to-origin geometry for each primary track segment."""
    starts = primary_df[["PosX_um", "PosY_um", "PosZ_um"]].to_numpy(float)
    stops = primary_df[["StopPosX_um", "StopPosY_um", "StopPosZ_um"]].to_numpy(float)
    segments = stops - starts
    segment_lengths_sq = np.sum(segments**2, axis=1)

    # Closest point to origin on finite segment A + t(B-A), with t clipped to [0, 1].
    t_closest = np.zeros(len(primary_df), dtype=float)
    nonzero = segment_lengths_sq > 0
    t_closest[nonzero] = np.clip(
        -np.sum(starts[nonzero] * segments[nonzero], axis=1) / segment_lengths_sq[nonzero],
        0.0,
        1.0,
    )

    closest_points = starts + t_closest[:, None] * segments
    closest_distance_um = np.linalg.norm(closest_points, axis=1)

    enriched = primary_df.copy()
    enriched["HasDamage_GetAllParticle"] = enriched["Particle_ID"].isin(damaging_primary_ids)
    enriched["TrackLength_um"] = np.sqrt(segment_lengths_sq)
    enriched["ClosestApproachT"] = t_closest
    enriched["ClosestApproachX_um"] = closest_points[:, 0]
    enriched["ClosestApproachY_um"] = closest_points[:, 1]
    enriched["ClosestApproachZ_um"] = closest_points[:, 2]
    enriched["ClosestDistanceToOrigin_um"] = closest_distance_um

    if nucleus_radius_um is not None:
        enriched["CrossesNucleus"] = enriched["ClosestDistanceToOrigin_um"] <= nucleus_radius_um

    return enriched


# Set this to your nucleus radius if you want a true/false crossing flag.
nucleus_radius_um = None

primary_geometry_df = add_track_origin_geometry(
    primary_df,
    damaging_primary_ids=getall_style_ids,
    nucleus_radius_um=nucleus_radius_um,
)

primary_geometry_df.groupby("HasDamage_GetAllParticle")["ClosestDistanceToOrigin_um"].describe()

,count,mean,std,min,25%,50%,75%,max
HasDamage_GetAllParticle,,,,,,,,
False,641.0,3.959155,0.562509,2.841107,3.459473,3.962427,4.450514,4.987567
True,259.0,2.113445,0.692342,0.273541,1.647337,2.239143,2.710491,3.074339
